<a href="https://colab.research.google.com/github/PNV-do-an/AI-Interview-Platform-BE/blob/main/docs/colab-deploy-notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Deploy AI Avatar Interview - Google Colab (Free Tier)

**Hệ thống phỏng vấn video call người thật bằng AI LiveTalking**

## ⚡ Quick Start
Click ▶️ Run vào từng cell theo thứ tự từ trên xuống dưới.

### ⚠️ BẮT BUỘC:
- **Cell 5 cần ngrok authtoken**: Lấy từ https://dashboard.ngrok.com/get-started/your-authtoken
- **Hardware**: Đảm bảo runtime có GPU (Runtime → Change runtime type → T4 GPU)

In [ ]:
# CELL 1: Cài đặt dependencies và cấu hình Docker
!sudo apt-get update > /dev/null 2>&1
!sudo apt-get install -y docker.io curl > /dev/null 2>&1
!curl -fsSL https://get.docker.com | sh > /dev/null 2>&1

import subprocess
import json

docker_config = {
    "exec": ["json"],
    "runtimes": {
        "nvidia": {
            "path": "nvidia-container-runtime",
            "arguments": []
        }
    }
}

with open('/etc/docker/daemon.json', 'w') as f:
    json.dump(docker_config, f)

subprocess.run(['sudo', 'systemctl', 'restart', 'docker'])

import time
time.sleep(10)

print("Checking GPU...")
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print("✅ GPU detected!")
else:
    print("⚠️ No GPU detected")

!pip install pyngrok requests python-dotenv aiohttp uvicorn faster-whisper imageio-ffmpeg numpy edge-tts python-multipart > /dev/null 2>&1

print("Setup completed!")

In [ ]:
# CELL 2: Clone repository AI Service
!git clone https://github.com/PNV-do-an/AI-Interview-Platform-AI.git
%cd AI-Interview-Platform-AI

import os
print(f"Current directory: {os.getcwd()}")

if os.path.exists('Dockerfile'):
    print("✅ Found Dockerfile!")
else:
    print("❌ No Dockerfile found")

In [ ]:
# CELL 3: Build Docker Image (~10-15 phút)
print("Building Docker image...")
!docker build -t hiremind-ai-service .

print("Docker image built successfully!")
!docker images hiremind-ai-service --format "{{.Repository}}:{{.Tag}} - Size: {{.Size}}"

In [ ]:
# CELL 4: Chạy container với port mapping
print("Starting AI service container...")
!docker rm -f hiremind-ai > /dev/null 2>&1 || true

!docker run -d \
  --name hiremind-ai \
  --gpus all \
  -p 8010:8010 \
  -p 8020:8020 \
  -e AI_AVATAR_SERVER=http://livetalking:8010 \
  -e AI_ALLOWED_ORIGIN=* \
  hiremind-ai-service

import time
print("Waiting for services to warm up...")
time.sleep(35)

!docker logs hiremind-ai --tail 20

import requests
try:
    response = requests.get('http://localhost:8020/health', timeout=10)
    if response.status_code == 200:
        data = response.json()
        print("✅ Health check passed!")
        print(f"STT Model: {data.get('sttModel')}")
    else:
        print(f"Health check failed: {response.status_code}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# CELL 5: Tạo ngrok tunnel (YÊU CẦN NGROK AUTHTOKEN!)
from pyngrok import ngrok
import os

NGROK_AUTH_TOKEN = 'YOUR_NGROK_AUTHTOKEN_HERE'

if NGROK_AUTH_TOKEN == 'YOUR_NGROK_AUTHTOKEN_HERE':
    print("ERROR: Replace YOUR_NGROK_AUTHTOKEN_HERE with your actual token!")
    raise ValueError("Ngrok auth token is required!")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("Creating ngrok tunnel to port 8020...")
public_url = ngrok.connect(8020)
print(f"AI Service URL: {public_url.public_url}")

print("Creating ngrok tunnel to port 8010 (WebRTC)...")
webrtc_url = ngrok.connect(8010)
print(f"WebRTC Stream URL: {webrtc_url.public_url}")

print("=" * 70)
print("COPI Y THESE URLs!")
print("=" * 70)
print(f"AI_SERVICE_URL={public_url.public_url}")
print(f"WEBRTC_URL={webrtc_url.public_url}")
print("=" * 70)

os.environ['AI_SERVICE_URL'] = public_url.public_url
os.environ['WEBRTC_URL'] = webrtc_url.public_url

In [ ]:
# CELL 6: Test endpoints
import requests
import os

base_url = os.environ.get('AI_SERVICE_URL', 'http://localhost:8020').replace('https://', '')

tests_passed = 0

print("Testing endpoints...")

resp = requests.get(f'http://{base_url}/health', timeout=10)
if resp.status_code == 200:
    print("✅ GET /health PASSED")
    tests_passed += 1
else:
    print(f"❌ GET /health FAILED: {resp.status_code}")

resp = requests.get(f'http://{base_url}/api/admin/sessions', timeout=10)
if resp.status_code == 200:
    print("✅ GET /api/admin/sessions PASSED")
    tests_passed += 1
else:
    print(f"❌ GET /api/admin/sessions FAILED: {resp.status_code}")

print(f"\nResults: {tests_passed}/2 tests passed")

In [ ]:
# CELL 7: Frontend Integration Guide
import os

ai_service_url = os.environ.get('AI_SERVICE_URL', 'N/A')

print("Frontend Configuration:")
print("-" * 70)
print(f"REACT_APP_API_URL=https://aiinterview-vn.onrender.com/api/v1")
print(f"REACT_APP_AI_SERVICE_URL={ai_service_url}")
print("-" * 70)

print("\nUpdate Vercel Environment Variables:")
print("1. Go to: https://vercel.com/dashboard")
print("2. Settings → Environment Variables")
print("3. Add REACT_APP_AI_SERVICE_URL=<above_url>")
print("4. Redeploy: vercel deploy --prod")

In [ ]:
# CELL 8: Troubleshooting & Monitoring
print("Container Resources:")
!docker stats --no-stream hiremind-ai

print("\nGPU Usage:")
!nvidia-smi

print("\nLogs:")
!docker logs hiremind-ai --tail 15

print("\nQuick fixes:")
print("• If GPU not detected: Runtime → Factory reset → Select T4 GPU")
print("• If Docker fails: Re-run Cell 1")
print("• If ngrok error: Verify token format (2xXXXXXXXXX)")